# Create Multiscale Zarr Pyramid for Web Visualization

This notebook builds a multiscale Zarr pyramid from the MODIS snow phenology Icechunk store so the dataset can be visualized as a slippy web map using [zarr-layer](https://github.com/carbonplan/zarr-layer) without creating a separate visualization copy or running a tile server.

**Stack**
- [topozarr](https://github.com/carbonplan/topozarr): generates coarsened multi-resolution levels following the [GeoZarr multiscales spec](https://github.com/zarr-developers/geozarr-spec)
- [zarr-layer](https://github.com/carbonplan/zarr-layer): TypeScript library that fetches and renders Zarr as a custom MapLibre/Mapbox layer, reprojecting on the GPU on the fly

**Why multiscales?**  
The full store is 86 400 × 43 200 pixels. Without overview levels, zarr-layer would need to fetch the entire array at every zoom level. Multiscales let the viewer fetch only the appropriate coarsened level for the current viewport — qualitatively matching the performance of a traditional tile server.

**Projection note**  
The store is in MODIS Sinusoidal, *not* Web Mercator. zarr-layer reprojects on the GPU client-side, so we can write the pyramid in native projection and preserve pixel fidelity.

---
**Dependencies**: `topozarr` and `xproj` are added to `pixi.toml` as PyPI dependencies. Run `pixi install` before launching this notebook.

## Imports

In [1]:
from pathlib import Path
import sys
import xarray as xr
import zarr
import icechunk
import rioxarray
import xproj
from topozarr import create_pyramid, ZarrLayerVarConfig

sys.path.insert(0, str(Path('..').resolve()))
from modis_snow_phenology import Config

config = Config('config/config_v1.txt')

In [ ]:
# # try example on topozarr README.md
ds = xr.tutorial.open_dataset('air_temperature', chunks="auto", mask_and_scale=False) # mask_and_scale=False
ds = ds.proj.assign_crs(spatial_ref="EPSG:4326")
ds

In [ ]:
pyramid = create_pyramid(
    ds,
    levels=2,
    x_dim="lon",
    y_dim="lat",
    method="mean",  # "mean" (default) | "max" | "min" | "sum"
)
pyramid

In [ ]:
print(pyramid.encoding)

In [ ]:
pyramid.dt['/1'].ds['air'].encoding

# encoding with mask_and_scale=True
# {'dtype': dtype('int16'),
#  'source': '/home/eric/.cache/xarray_tutorial_data/69c68be1605878a6c8efdd34d85b4ca1-air_temperature.nc',
#  'original_shape': (2920, 25, 53),
#  'scale_factor': np.float64(0.01)}

# encoding with mask_and_scale=False
# {'dtype': dtype('int16'),
#  'source': '/home/eric/.cache/xarray_tutorial_data/69c68be1605878a6c8efdd34d85b4ca1-air_temperature.nc',
#  'original_shape': (2920, 25, 53)}

In [ ]:
pyramid.dt['/1'].ds # this still looks lazy, that's good right???

# access to original dataset in the datatree
level_0_ds = pyramid.dt['/0'].ds.isel(time=0)
level_1_ds = pyramid.dt['/1'].ds.isel(time=0)

import matplotlib.pyplot as plt
f,ax=plt.subplots(1,2, figsize=(12,6))
level_0_ds.air.plot(ax=ax[0], vmin=200, vmax=300)
ax[0].set_title('Level 0')
level_1_ds.air.plot(ax=ax[1], vmin=200, vmax=300)
ax[1].set_title('Level 1')
f.tight_layout()

In [ ]:
for level in pyramid.dt:
    print(f"Level: {level}")
    print(pyramid.dt[level].ds.dims)
    print(f"variable air's encoding is: {pyramid.dt[level].ds['air'].encoding}")
    #print(pyramid.dt[level].ds)

## 1. Open the Icechunk Store

In [2]:
storage = icechunk.azure_storage(
    account=config.azure_storage_account,
    container=config.azure_container,
    prefix=config.icechunk_prefix,
    sas_token=config.azure_storage_sas_token,
)
repo = icechunk.Repository.open(storage)
session = repo.readonly_session('main')

ds = xr.open_zarr(session.store, zarr_format=3, consolidated=False, decode_coords='all',mask_and_scale=False)
ds

<xarray.Dataset> Size: 224GB
Dimensions:               (water_year: 10, y: 43200, x: 86400)
Coordinates:
  * water_year            (water_year) int64 80B 2015 2016 2017 ... 2023 2024
  * y                     (y) float64 346kB 1.001e+07 1.001e+07 ... -1.001e+07
  * x                     (x) float64 691kB -2.001e+07 -2.001e+07 ... 2.001e+07
    spatial_ref           int64 8B ...
Data variables:
    SAD_DOWY              (water_year, y, x) int16 75GB dask.array<chunksize=(1, 600, 600), meta=np.ndarray>
    max_consec_snow_days  (water_year, y, x) int16 75GB dask.array<chunksize=(1, 600, 600), meta=np.ndarray>
    SDD_DOWY              (water_year, y, x) int16 75GB dask.array<chunksize=(1, 600, 600), meta=np.ndarray>
Attributes:
    title:        Global MODIS Snow Phenology
    description:  Snow appearance date (SAD), snow disappearance date (SDD), ...
    source:       MODIS MOD10A2.061 via Microsoft Planetary Computer
    Conventions:  CF-1.8

## 2. Prepare Dataset

topozarr needs:
1. A CRS assigned via `xproj` (`.proj.assign_crs()`)
2. The `spatial_ref` scalar coordinate removed — topozarr manages CRS metadata internally and the scalar causes issues during coarsening

Data comes out of the store as `float32` (Zarr's `mask_and_scale` replaces `int16` fill values with `NaN`). `create_pyramid(method='mean')` propagates NaNs correctly, so no extra masking is needed.

In [3]:
# Extract CRS WKT from the rioxarray spatial_ref before we drop it
crs = ds.rio.crs
print('Native CRS:', crs)
print()
print('WKT:')
print(crs.to_wkt())

Native CRS: PROJCS["unnamed",GEOGCS["Unknown datum based upon the custom spheroid",DATUM["Not specified (based on custom spheroid)",SPHEROID["Custom spheroid",6371007.181,0]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Sinusoidal"],PARAMETER["longitude_of_center",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["Meter",1],AXIS["Easting",EAST],AXIS["Northing",NORTH]]

WKT:
PROJCS["unnamed",GEOGCS["Unknown datum based upon the custom spheroid",DATUM["Not specified (based on custom spheroid)",SPHEROID["Custom spheroid",6371007.181,0]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Sinusoidal"],PARAMETER["longitude_of_center",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["Meter",1],AXIS["Easting",EAST],AXIS["Northing",NORTH]]


In [4]:
# Drop the CF-convention spatial_ref scalar — xproj will carry the CRS instead
ds_clean = ds.drop_vars('spatial_ref')

# Assign CRS via xproj so topozarr can find it
ds_crs = ds_clean.proj.assign_crs(spatial_ref_crs={'wkt': crs.to_wkt()})

ds_crs

<xarray.Dataset> Size: 224GB
Dimensions:               (water_year: 10, y: 43200, x: 86400)
Coordinates:
  * water_year            (water_year) int64 80B 2015 2016 2017 ... 2023 2024
  * y                     (y) float64 346kB 1.001e+07 1.001e+07 ... -1.001e+07
  * x                     (x) float64 691kB -2.001e+07 -2.001e+07 ... 2.001e+07
  * wkt                   int64 8B 0
Data variables:
    SAD_DOWY              (water_year, y, x) int16 75GB dask.array<chunksize=(1, 600, 600), meta=np.ndarray>
    max_consec_snow_days  (water_year, y, x) int16 75GB dask.array<chunksize=(1, 600, 600), meta=np.ndarray>
    SDD_DOWY              (water_year, y, x) int16 75GB dask.array<chunksize=(1, 600, 600), meta=np.ndarray>
Indexes:
    wkt      CRSIndex (crs=PROJCS["unnamed",GEOGCS["Unknown datum based upon the custom sphero ...)
Attributes:
    title:        Global MODIS Snow Phenology
    description:  Snow appearance date (SAD), snow disappearance date (SDD), ...
    source:       MODIS MOD10A2.061 via Microsoft Planetary Computer
    Conventions:  CF-1.8

## 3. Create Multiscale Pyramid

### Choosing the number of levels

Each level coarsens by 2× in both spatial dimensions. The coarsest level (level 0 in topozarr's convention) should be small enough to fetch in a single request at the lowest zoom.

| Levels | Coarsest shape (y × x) | Coarsest pixel size |
|--------|------------------------|---------------------|
| 7      | 337 × 675              | ~59 km              |
| 8      | 169 × 337              | ~118 km             |
| 9      | 84 × 169               | ~236 km             |

8 levels gives a manageable coarsest tile (~169 × 337 pixels) while still preserving meaningful spatial detail at the finest level.

### Coarsening method
`method='mean'` is used for all three variables. For day-of-year metrics (SAD, SDD) this is a spatial average, which is visually reasonable. Override with `'min'` or `'max'` if you want conservative estimates.

### Non-spatial dimension
The `water_year` dimension is preserved at every pyramid level — zarr-layer handles it as a non-spatial dimension that the user can slice interactively.

In [5]:
# Optional: supply colormap and range hints for zarr-layer
# SAD/SDD range is 1–366 (day of water year); max_consec is 0–366
layer_hints = {
    'SAD_DOWY': ZarrLayerVarConfig(clim=[1, 366],  colormap='Purples'),
    'SDD_DOWY': ZarrLayerVarConfig(clim=[1, 366],  colormap='Reds'),
    'max_consec_snow_days': ZarrLayerVarConfig(clim=[1, 366], colormap='Blues'), 
}

In [6]:
N_LEVELS = 5

pyramid = create_pyramid(
    ds_crs,
    levels=N_LEVELS,
    x_dim='x',
    y_dim='y',
    method='mean',
    target_chunk_bytes=int(0.5 * 1024 * 1024),  # 500 KB chunks — web-friendly
    chunks_per_shard=4,
    layer_hints=layer_hints,
)

pyramid

Pyramid(datatree=<xarray.DataTree 'root'>
Group: /
│   Attributes:
│       zarr_conventions:    [{'schema_url': 'https://raw.githubusercontent.com/z...
│       multiscales:         {'layout': [{'asset': '0', 'transform': {'scale': [1...
│       proj:code:           PROJCS["unnamed",GEOGCS["Unknown datum based upon th...
│       spatial:dimensions:  ['y', 'x']
│       spatial:transform:   [463.3127165287733, 0.0, -20015109.354005992, 0.0, -...
│       spatial:bbox:        [-20015109.354005992, -10007554.677040007, 20015109....
│       spatial:shape:       [43200, 86400]
│       zarr-layer:          {'SAD_DOWY': {'clim': [1, 366], 'colormap': 'Purples...
├── Group: /0
│       Dimensions:               (water_year: 10, y: 43200, x: 86400)
│       Coordinates:
│         * water_year            (water_year) int64 80B 2015 2016 2017 ... 2023 2024
│         * y                     (y) float64 346kB 1.001e+07 1.001e+07 ... -1.001e+07
│         * x                     (x) float64 691kB -2.001e+

In [7]:
for level in pyramid.dt:
    print(f"Level: {level}")
    print(pyramid.dt[level].ds.dims)
    print(f"variable SAD_DOWY encoding is: {pyramid.dt[level].ds['SAD_DOWY'].encoding}")
    print(f"variable SDD_DOWY encoding is: {pyramid.dt[level].ds['SDD_DOWY'].encoding}")
    print(f"variable max_consec_snow_days encoding is: {pyramid.dt[level].ds['max_consec_snow_days'].encoding}")

Level: 0
FrozenMappingWarningOnValuesAccess({'water_year': 10, 'y': 43200, 'x': 86400})
variable SAD_DOWY encoding is: {'chunks': (1, 600, 600), 'preferred_chunks': {'water_year': 1, 'y': 600, 'x': 600}, 'compressors': (BloscCodec(_tunable_attrs=set(), typesize=2, cname=<BloscCname.zstd: 'zstd'>, clevel=5, shuffle=<BloscShuffle.shuffle: 'shuffle'>, blocksize=0),), 'filters': (), 'shards': (1, 2400, 2400), 'serializer': BytesCodec(endian=<Endian.little: 'little'>), 'dtype': dtype('<i2'), 'grid_mapping': 'spatial_ref'}
variable SDD_DOWY encoding is: {'chunks': (1, 600, 600), 'preferred_chunks': {'water_year': 1, 'y': 600, 'x': 600}, 'compressors': (BloscCodec(_tunable_attrs=set(), typesize=2, cname=<BloscCname.zstd: 'zstd'>, clevel=5, shuffle=<BloscShuffle.shuffle: 'shuffle'>, blocksize=0),), 'filters': (), 'shards': (1, 2400, 2400), 'serializer': BytesCodec(endian=<Endian.little: 'little'>), 'dtype': dtype('<i2'), 'grid_mapping': 'spatial_ref'}
variable max_consec_snow_days encoding is:

## 4. Write Pyramid to Zarr

Two options are shown below:
- **Local**: write to `notebooks/data/` for local inspection or a local HTTP server
- **Azure**: write to the `uwcryo` storage account under a new prefix so zarr-layer can reach it from the browser

The DataTree hierarchy maps directly to Zarr group hierarchy:
```
modis_snow_phenology_multiscale.zarr/
├── .zattrs          ← multiscales + layer-hints metadata
├── 0/               ← coarsest level (169 × 337)
├── 1/
├── ...
└── 7/               ← native resolution (43200 × 86400)
```

### 4a. Write locally

In [ ]:
# LOCAL_OUT = Path('../data/modis_snow_phenology_multiscale.zarr')
# LOCAL_OUT.parent.mkdir(parents=True, exist_ok=True)

# # DataTree.to_zarr writes each level as a Zarr group.
# # Pass pyramid.encoding so topozarr's recommended chunk/shard sizes are respected.
# pyramid.dt.to_zarr(LOCAL_OUT, mode='w', encoding=pyramid.encoding)

# print(f'Written to {LOCAL_OUT.resolve()}')

In [ ]:
# Quick verification — open the coarsest level
# store_check = zarr.open(LOCAL_OUT, mode='r')
# print(store_check.tree())

### 4b. Write to Azure via Icechunk (recommended)

Use `icechunk.azure_storage` + `session.store` — the same pattern as the rest of the pipeline. This gives you a versioned, committable store on Azure that zarr-layer can read directly (the blog post confirms zarr-layer is compatible with Icechunk-backed stores).

For zarr-layer to reach the store from a browser the container must allow **anonymous blob reads** or use a SAS URL. Set the container access level to *Blob* in the Azure portal (or via `az storage container set-permission`).

In [8]:
MULTISCALE_PREFIX = 'modis_snow_phenology/modis_snow_phenology_multiscale_v1'

pyramid_storage = icechunk.azure_storage(
    account=config.azure_storage_account,
    container=config.azure_container,
    prefix=MULTISCALE_PREFIX,
    sas_token=config.azure_storage_sas_token,
)
pyramid_repo = icechunk.Repository.open_or_create(pyramid_storage)
session = pyramid_repo.writable_session('main')

pyramid.dt.to_zarr(session.store, mode='w', encoding=pyramid.encoding, consolidated=False)
session.commit('write multiscale pyramid: 5 levels, mean coarsening, WY2015-2024')

AZURE_URL = (
    f'https://{config.azure_storage_account}.blob.core.windows.net'
    f'/{config.azure_container}/{MULTISCALE_PREFIX}'
)
print(f'Written to: {AZURE_URL}')

: 

## 5. Visualize with zarr-layer

zarr-layer is a TypeScript library — the instructions below describe how to wire it into a MapLibre page.  
See the [zarr-layer demo](https://zarr-layer-demo.vercel.app/) and [repo](https://github.com/carbonplan/zarr-layer) for live examples.

### Install
```bash
npm install @carbonplan/zarr-layer
# or via CDN in a plain HTML file:
# <script type="module">import { ZarrLayer } from 'https://cdn.jsdelivr.net/npm/@carbonplan/zarr-layer/+esm'</script>
```

### Minimal MapLibre example

Replace `ZARR_URL` with your public Azure URL or a local `http://localhost:8080/modis_snow_phenology_multiscale.zarr`.

```html
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8" />
  <link href="https://unpkg.com/maplibre-gl/dist/maplibre-gl.css" rel="stylesheet" />
  <style> body { margin: 0; } #map { width: 100vw; height: 100vh; } </style>
</head>
<body>
  <div id="map"></div>
  <script type="module">
    import maplibregl from 'https://cdn.skypack.dev/maplibre-gl';
    import { ZarrLayer } from 'https://cdn.jsdelivr.net/npm/@carbonplan/zarr-layer/+esm';

    const ZARR_URL = 'https://uwcryo.blob.core.windows.net/snowmelt/modis_snow_phenology/modis_snow_phenology_v1_multiscale';

    const map = new maplibregl.Map({
      container: 'map',
      style: 'https://demotiles.maplibre.org/style.json',
      center: [0, 30],
      zoom: 2,
    });

    map.on('load', async () => {
      const layer = await ZarrLayer.create({
        id: 'snow-phenology',
        source: ZARR_URL,
        variable: 'SDD_DOWY',       // 'SAD_DOWY' or 'max_consec_snow_days'
        selector: { water_year: 0 }, // index into the water_year dimension (0 = WY2015)
        colormap: 'Blues',
        clim: [1, 366],
        opacity: 0.8,
      });

      map.addLayer(layer);
    });
  </script>
</body>
</html>
```

### Serve locally for testing

zarr-layer fetches chunks via HTTP range requests. To test the local zarr store:

```bash
# From the data/ directory:
python -m http.server 8080
```

Then set `ZARR_URL = 'http://localhost:8080/modis_snow_phenology_multiscale.zarr'`.
You may need to add CORS headers — use `npx http-server --cors` instead of Python's server.

### Slicing across water years

The `water_year` dimension is non-spatial and is preserved at every pyramid level. zarr-layer's `selector` parameter slices it by index:

```js
// WY2024 is index 9 (WY2015=0, ..., WY2024=9)
layer.setSelector({ water_year: 9 });
```

### Reprojection

The pyramid is in MODIS Sinusoidal. zarr-layer reprojects on the GPU using a mesh-based approach (no server-side reprojection needed). Specify the source projection if it is not detected automatically:

```js
const layer = await ZarrLayer.create({
  ...,
  sourceCrs: '+proj=sinu +R=6371007.181 +nadgrids=@null +wktext',
});
```